In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# =================================================================
# FONCTIONs D'ACTIVATION
# =================================================================

def sigmoid(x):
    """Fonction d'activation sigmoid"""
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

def identity(x):
  """Fonction d'activation identité"""
  return x

# =================================================================
# POIDS ET BIAIS FINAUX DE L'APPRENTISSAGE
# =================================================================

# Poids finaux obtenus après apprentissage
final_W1 = np.array([
    [ 0.39911345, -0.18617998,  0.20715008,  0.83885701, -0.14652933,  0.04371505],
    [ 1.10285481,  0.12614611, -0.48823469,  0.42913213, -0.30279306,  0.09582355]
])

final_W2 = np.array([
    [ 1.58526133],
    [-0.0949744 ],
    [-0.20437294],
    [ 0.96531223],
    [ 0.1657301 ],
    [ 1.0702865 ]
])

final_b1 = np.array([[ 0.23174949, -0.24686502, -0.23820052,  0.10958827, -0.07730999,  0.27197872]])
final_b2 = np.array([[1.85065906]])

print("=== POIDS ET BIAIS DU RÉSEAU ENTRAÎNÉ ===")
print(f"W1 shape: {final_W1.shape}")
print(f"W2 shape: {final_W2.shape}")
print(f"b1 shape: {final_b1.shape}")
print(f"b2 shape: {final_b2.shape}")

# =================================================================
# FONCTION DE PRÉDICTION
# =================================================================

def predict_rna_sigmoid(temps, croissance_precedente, W1, W2, b1, b2, scaler_X, scaler_y):
    """Prédiction avec sigmoid partout (comme l'entraînement)"""
    x_input = np.array([[temps, croissance_precedente]])
    x_normalized = scaler_X.transform(x_input)
    
    # Couche cachée avec sigmoid
    z1 = np.dot(x_normalized, W1) + b1
    a1 = sigmoid(z1)
    
    # Couche de sortie avec sigmoid aussi en cohérence avec l'apprentissage pour obtenir de meilleure résultat
    z2 = np.dot(a1, W2) + b2
    y_pred_normalized = sigmoid(z2)
    
    # Dénormaliser
    y_pred = scaler_y.inverse_transform(y_pred_normalized)
    return y_pred[0, 0]

def predict_multi_steps(initial_temps, initial_croissance, n_steps, W1, W2, b1, b2, scaler_X, scaler_y):
    """Prédiction à plusieurs pas"""
    predictions = []
    current_temps = initial_temps
    current_croissance = initial_croissance
    
    for step in range(n_steps):
        next_croissance = predict_rna_sigmoid(current_temps, current_croissance, W1, W2, b1, b2, scaler_X, scaler_y)
        predictions.append(next_croissance)
        current_temps += 1
        current_croissance = next_croissance
    
    return np.array(predictions)

# =================================================================
# CHARGEMENT ET PRÉPARATION DES DONNÉES
# =================================================================

url = "https://raw.githubusercontent.com/faniloo08/ANNPrediction/main/data/serie_temporelle.csv"
df = pd.read_csv(url, usecols=["temps", "croissance_A2"])

df = df[df['croissance_A2'] != float('-inf')]
df = df[df['croissance_A2'] != float('inf')]
df = df.dropna()
df = df.sort_values('temps')

print(f"\n=== DONNÉES POUR PRÉDICTION ===")
print(f"Nombre total d'échantillons: {len(df)}")
print(f"Plage des valeurs réelles: [{df['croissance_A2'].min():.8f}, {df['croissance_A2'].max():.8f}]")

# Préparer les données
X = []
y = []
for i in range(1, len(df)):
    X.append([df.iloc[i-1]['temps'], df.iloc[i-1]['croissance_A2']])
    y.append([df.iloc[i]['croissance_A2']])

X = np.array(X)
y = np.array(y)

# Split et normalisation (même que l'entraînement)
split_idx = int(0.8 * len(X))
X_train = X[:split_idx]
X_test = X[split_idx:]
y_train = y[:split_idx]
y_test = y[split_idx:]

scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
y_train_scaled = scaler_y.fit_transform(y_train)
X_test_scaled = scaler_X.transform(X_test)
y_test_scaled = scaler_y.transform(y_test)

# =================================================================
# ÉVALUATION DU MODÈLE SIGMOID
# =================================================================

print(f"\n=== ÉVALUATION DU MODÈLE SIGMOID ===")

# Test sur échantillons
test_predictions = []
test_actuals = []

for i in range(min(50, len(X_test))):  # Tester sur 50 échantillons max
    pred = predict_rna_sigmoid(X_test[i, 0], X_test[i, 1], final_W1, final_W2, final_b1, final_b2, scaler_X, scaler_y)
    test_predictions.append(pred)
    test_actuals.append(y_test[i, 0])

test_predictions = np.array(test_predictions)
test_actuals = np.array(test_actuals)

# Métriques
mae = mean_absolute_error(test_actuals, test_predictions)
rmse = np.sqrt(mean_squared_error(test_actuals, test_predictions))
if np.var(test_actuals) > 0:
    r2 = 1 - mean_squared_error(test_actuals, test_predictions) / np.var(test_actuals)
else:
    r2 = float('-inf')
mape = np.mean(np.abs((test_actuals - test_predictions) / np.where(test_actuals != 0, test_actuals, 1e-8))) * 100

# Vérifier si les prédictions sont dans l'intervalle souhaité [0.4, 0.7]
in_range = np.sum((test_predictions >= 0.4) & (test_predictions <= 0.7))
range_percentage = (in_range / len(test_predictions)) * 100

print(f"MAE: {mae:.8f}")
print(f"RMSE: {rmse:.8f}")
print(f"MAPE: {mape:.8f}%")
print(f"R²: {r2:.8f}")
print(f"Plage prédictions: [{test_predictions.min():.8f}, {test_predictions.max():.8f}]")
print(f"Dans intervalle [0.4-0.7]: {in_range}/{len(test_predictions)} ({range_percentage:.1f}%)")

# =================================================================
# PRÉDICTIONS MULTI-HORIZONS
# =================================================================

start_temps = X_train[-1, 0]
start_croissance = X_train[-1, 1]

print(f"\n=== PRÉDICTIONS MULTI-HORIZONS ===")
print(f"Point de départ - Temps: {start_temps}, Croissance: {start_croissance:.6f}")

horizons = [1, 3, 10, 20]
predictions_dict = {}

for horizon in horizons:
    pred = predict_multi_steps(start_temps, start_croissance, horizon, 
                              final_W1, final_W2, final_b1, final_b2, 
                              scaler_X, scaler_y)
    predictions_dict[horizon] = pred
    print(f"Prédiction à {horizon:2d} pas: {pred[-1]:.8f}")

# =================================================================
# VISUALISATION : 4 GRAPHIQUES SÉPARÉS
# =================================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Prédictions Multi-Horizons (Sigmoid)', fontsize=16, fontweight='bold')

colors = ['red', 'orange', 'purple', 'brown']
markers = ['o', 's', '^', 'D']

for idx, horizon in enumerate(horizons):
    row = idx // 2
    col = idx % 2
    ax = axes[row, col]
    
    # Données de référence (quelques points avant le point de départ)
    ref_start = max(0, len(X_train) - 20)
    ref_times = [X_train[i, 0] for i in range(ref_start, len(X_train))]
    ref_values = [y_train[i, 0] for i in range(ref_start, len(y_train))]
    
    # Tracer les données de référence
    ax.plot(ref_times, ref_values, 'b-', linewidth=2, alpha=0.7, label='Données historiques')
    
    # Tracer les prédictions
    pred_times = np.arange(start_temps + 1, start_temps + horizon + 1)
    ax.plot(pred_times, predictions_dict[horizon], 
            color=colors[idx], marker=markers[idx], linewidth=3, markersize=8,
            label=f'Prédiction {horizon} pas')
    
    # Ligne de séparation au point de départ
    ax.axvline(x=start_temps, color='black', linestyle='--', alpha=0.6, 
               label='Point de départ')
    
    # Zone cible [0.4, 0.7]
    ax.axhspan(0.4, 0.7, alpha=0.2, color='green', label='Zone cible [0.4-0.7]')
    
    # Mise en forme
    ax.set_title(f'Prédiction à {horizon} pas\n(Valeur finale: {predictions_dict[horizon][-1]:.4f})', 
                fontsize=14, fontweight='bold')
    ax.set_xlabel('Temps', fontsize=12)
    ax.set_ylabel('Croissance A2', fontsize=12)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # Ajuster les limites y pour mieux voir
    all_values = ref_values + predictions_dict[horizon].tolist()
    y_min = min(all_values) - 0.1
    y_max = max(all_values) + 0.1
    ax.set_ylim(y_min, y_max)

plt.tight_layout()
plt.show()

# =================================================================
# RÉSUMÉ FINAL
# =================================================================

print(f"\n=== RÉSUMÉ FINAL ===")
print(f"Modèle: Sigmoid-Sigmoid")
print(f"Test sur {len(test_predictions)} échantillons")
print(f"MAE: {mae:.8f}")
print(f"RMSE: {rmse:.8f}")
print(f"MAPE: {mape:.8f}%")
print(f"R²: {r2:.8f}")
print(f"Prédictions dans [0.4-0.7]: {range_percentage:.1f}%")

print(f"\n=== CRITÈRES D'ÉVALUATION D'UNE BONNE PRÉDICTION ===")
print(f"MAPE < 20% : {'OUI' if mape < 20 else 'NON'} ({mape:.1f}%)")
print(f"R² > 0.5 : {'OUI' if r2 > 0.5 else 'NON'} ({r2:.3f})")
print(f"Prédictions dans [0.4-0.7] > 80% : {'OUI' if range_percentage > 80 else 'NON'} ({range_percentage:.1f}%)")

# Résultats d'entraînement du réseau de neurones

## Poids et biais du réseau entraîné

| Paramètre | Dimensions |
|-----------|------------|
| **W1**    | (2, 6)     |
| **W2**    | (6, 1)     |
| **b1**    | (1, 6)     |
| **b2**    | (1, 1)     |

## Données pour prédiction

- **Nombre total d'échantillons :** 501
- **Plage des valeurs réelles :** [0.10000000, 0.50000000]

## Évaluation du modèle Sigmoid

### Métriques de performance
| Métrique | Valeur |
|----------|--------|
| **MAE**  | 0.00327785 |
| **RMSE** | 0.00327794 |
| **MAPE** | 0.65556976% |
| **R²**   | -inf |

### Analyse des prédictions
- **Plage des prédictions :** [0.49668073, 0.49676254]
- **Prédictions dans l'intervalle [0.4-0.7] :** 50/50 (100.0%)

## Prédictions multi-horizons

**Point de départ :** Temps = 399.0, Croissance = 0.500000

| Horizon | Prédiction |
|---------|------------|
| 1 pas   | 0.49667900 |
| 3 pas   | 0.49667362 |
| 10 pas  | 0.49668579 |
| 20 pas  | 0.49670294 |

![prédictions](predictions_A2.png)

## Résumé final

- **Modèle :** Sigmoid-Sigmoid
- **Test sur :** 50 échantillons
- **MAE :** 0.00327785
- **RMSE :** 0.00327794
- **MAPE :** 0.65556976%
- **R² :** -inf
- **Prédictions dans [0.4-0.7] :** 100.0%

## Critères d'évaluation d'une bonne prédiction

| Critère | Seuil | Résultat | Statut |
|---------|-------|----------|--------|
| **MAPE < 20%** | < 20% | 0.7% |  **OUI** |
| **R² > 0.5** | > 0.5 | -inf |  **NON** |
| **Prédictions dans [0.4-0.7] > 80%** | > 80% | 100.0% |  **OUI** |